# What's Inside a Model?

**Time:** ~20 minutes   
**Goal:** Download a real model and look at every piece inside it — no libraries, just raw bytes.

---

So far we used `transformers` and PyTorch to run GPT-2 as a black box.  
Today we crack it open.

A model file is just a **container of numbered arrays** (tensors).  
Think of it like a tarball full of CSV files — except the numbers are in binary and there's a JSON header telling you what's what.

## What Is a Model, Really?

If you've ever deployed a Docker image, you know the image isn't the application — it's **layers of files** that, combined, form the application.

A model is similar. It's not a program. It's a **collection of weight matrices** (giant grids of numbers) that were produced by training. The "inference engine" (vLLM, TensorRT-LLM, etc.) is the program that reads these matrices and does math with them.

```
┌─────────────────────────────────────────┐
│  Docker Image  =  Layers of files       │
│  Model File    =  Collection of tensors  │
│  Inference Eng  =  Program that runs it  │
└─────────────────────────────────────────┘
```

Today we download GPT-2 (124M parameters) and read every tensor by hand.

## But First — What Is a Tensor?

"Tensor" sounds intimidating, but it's just the generic word for an **N-dimensional array of numbers**:

```
Scalar          0 dimensions    42               a single number
Vector          1 dimension     [1.0, 2.0, 3.0]  a list of numbers
Matrix          2 dimensions    [[1, 2],          a grid / table
                                 [3, 4]]
3D Tensor       3 dimensions    a stack of grids  like a spreadsheet workbook
```

If you know numpy arrays, you already know tensors. A `numpy.ndarray` with shape `(50257, 768)` is a 2D tensor — a matrix with 50,257 rows and 768 columns. That's literally what the model's embedding table is.

**The word "tensor" is just the ML community's way of saying "an array of numbers with a shape."** In this notebook we use numpy on CPU, so they're just regular `ndarray`s. On a GPU they'd be PyTorch tensors or CUDA arrays. Same data, same shape, same math — different hardware.

## Step 1: Download GPT-2 from Hugging Face

Hugging Face is like Docker Hub for models. Each model repo contains:
- **Weight files** (`.safetensors`) — the actual numbers
- **Config** (`config.json`) — architecture: how many layers, dimensions, heads
- **Tokenizer files** — vocabulary and merge rules (we'll use these in notebook 02)

We'll download just the files we need — no `transformers` library, just HTTP requests.

In [1]:
import urllib.request
import os
import json

MODEL_DIR = "gpt2_weights"
os.makedirs(MODEL_DIR, exist_ok=True)

BASE_URL = "https://huggingface.co/openai-community/gpt2/resolve/main"

files_to_download = [
    "model.safetensors",
    "config.json",
    "vocab.json",
    "merges.txt",
]

for fname in files_to_download:
    path = os.path.join(MODEL_DIR, fname)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  Already exists: {fname} ({size_mb:.1f} MB)")
    else:
        url = f"{BASE_URL}/{fname}"
        print(f"  Downloading: {fname} ...", end=" ", flush=True)
        urllib.request.urlretrieve(url, path)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"done ({size_mb:.1f} MB)")

print(f"\nFiles in {MODEL_DIR}/:")
for f in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(os.path.join(MODEL_DIR, f))
    print(f"  {f:30s} {size:>12,} bytes")

  Downloading: model.safetensors ... done (522.7 MB)
  Downloading: config.json ... done (0.0 MB)
  Downloading: vocab.json ... done (1.0 MB)
  Downloading: merges.txt ... done (0.4 MB)

Files in gpt2_weights/:
  config.json                             665 bytes
  merges.txt                          456,318 bytes
  model.safetensors               548,105,171 bytes
  vocab.json                        1,042,301 bytes


## Step 2: Read the Config

Before touching the weights, let's read the blueprint. `config.json` tells us the model's architecture — like a Kubernetes deployment spec tells you the container image, replicas, and resource limits.

```
config.json    <=>    deployment.yaml
n_layer: 12    <=>    replicas: 12
n_embd: 768    <=>    memory: 768Mi
n_head: 12     <=>    cpu: 12
```

In [2]:
with open(os.path.join(MODEL_DIR, "config.json")) as f:
    config = json.load(f)

# The fields that matter for inference
print("GPT-2 Architecture")
print("=" * 45)
print(f"  n_layer  (transformer blocks) : {config['n_layer']}")
print(f"  n_embd   (embedding dimension) : {config['n_embd']}")
print(f"  n_head   (attention heads)     : {config['n_head']}")
print(f"  vocab_size                     : {config['vocab_size']}")
print(f"  n_positions (max seq length)   : {config['n_positions']}")
print(f"  activation_function            : {config.get('activation_function', 'gelu')}")
print()
print("What these mean:")
print(f"  - {config['n_layer']} transformer blocks stacked in series")
print(f"  - Each token is a vector of {config['n_embd']} numbers")
print(f"  - Each block has {config['n_head']} attention heads (head_dim = {config['n_embd'] // config['n_head']})")
print(f"  - Vocabulary of {config['vocab_size']} possible tokens")
print(f"  - Can handle sequences up to {config['n_positions']} tokens")

GPT-2 Architecture
  n_layer  (transformer blocks) : 12
  n_embd   (embedding dimension) : 768
  n_head   (attention heads)     : 12
  vocab_size                     : 50257
  n_positions (max seq length)   : 1024
  activation_function            : gelu_new

What these mean:
  - 12 transformer blocks stacked in series
  - Each token is a vector of 768 numbers
  - Each block has 12 attention heads (head_dim = 64)
  - Vocabulary of 50257 possible tokens
  - Can handle sequences up to 1024 tokens


## Step 3: Parse SafeTensors by Hand

SafeTensors is the standard format for storing model weights. It's simple:

```
┌──────────┬───────────────────────┬────────────────────┐
│ 8 bytes  │  N bytes              │  Rest of file      │
│ header   │  JSON header          │  Raw tensor data   │
│ length   │  (name, shape, dtype, │  (just floats,     │
│ (uint64) │   byte offsets)       │   packed tight)    │
└──────────┴───────────────────────┴────────────────────┘
```

It's like a tar file: a header that says "file X starts at byte Y, is Z bytes long" followed by the raw data.

We'll parse this with just `struct` (Python's binary reader) and `numpy` — no safetensors library.

In [3]:
import struct
import numpy as np

def parse_safetensors(path):
    """Parse a .safetensors file by hand. Returns dict of name -> numpy array."""
    with open(path, "rb") as f:
        # First 8 bytes = length of the JSON header
        header_len = struct.unpack("<Q", f.read(8))[0]
        
        # Read and parse the JSON header
        header = json.loads(f.read(header_len))
        
        # Everything after the header is raw tensor data
        data_start = 8 + header_len
        
        tensors = {}
        for name, info in header.items():
            print("name = ", name, "info = ", info)
            if name == "__metadata__":
                continue
            
            dtype_map = {"F32": np.float32, "F16": np.float16}
            dtype = dtype_map[info["dtype"]]
            shape = info["shape"]
            start, end = info["data_offsets"]
            
            f.seek(data_start + start)
            raw = f.read(end - start)
            tensors[name] = np.frombuffer(raw, dtype=dtype).reshape(shape)
        
        return tensors

weights = parse_safetensors(os.path.join(MODEL_DIR, "model.safetensors"))

print(f"Parsed {len(weights)} tensors from model.safetensors")
print(f"No safetensors library used — just struct.unpack + numpy.")

name =  __metadata__ info =  {'format': 'pt'}
name =  h.3.ln_2.bias info =  {'dtype': 'F32', 'shape': [768], 'data_offsets': [202420224, 202423296]}
name =  h.10.ln_1.weight info =  {'dtype': 'F32', 'shape': [768], 'data_offsets': [223154176, 223157248]}
name =  h.2.attn.c_attn.weight info =  {'dtype': 'F32', 'shape': [768, 2304], 'data_offsets': [541012992, 548090880]}
name =  h.4.attn.c_proj.weight info =  {'dtype': 'F32', 'shape': [768, 768], 'data_offsets': [18954240, 21313536]}
name =  h.7.attn.bias info =  {'dtype': 'F32', 'shape': [1, 1, 1024, 1024], 'data_offsets': [263339008, 267533312]}
name =  h.4.attn.c_attn.bias info =  {'dtype': 'F32', 'shape': [2304], 'data_offsets': [150975488, 150984704]}
name =  h.2.attn.bias info =  {'dtype': 'F32', 'shape': [1, 1, 1024, 1024], 'data_offsets': [516081664, 520275968]}
name =  h.5.mlp.c_proj.bias info =  {'dtype': 'F32', 'shape': [768], 'data_offsets': [21313536, 21316608]}
name =  h.1.attn.c_proj.weight info =  {'dtype': 'F32', 'shape

## Step 4: Explore Every Weight Tensor

Let's look at what we got. Each tensor has a **name** (like a file path) and a **shape** (its dimensions).

The naming convention tells you exactly where each tensor lives in the architecture:

```
wte.weight           = token embedding table
wpe.weight           = position embedding table
h.0.attn.c_attn.*    = layer 0, attention QKV projection
h.0.attn.c_proj.*    = layer 0, attention output projection
h.0.mlp.c_fc.*       = layer 0, FFN up-projection
h.0.mlp.c_proj.*     = layer 0, FFN down-projection
h.0.ln_1.*           = layer 0, first layer norm
h.0.ln_2.*           = layer 0, second layer norm
ln_f.*               = final layer norm
```

Think of it like a filesystem:
```
/model
  /embeddings
    wte.weight          [50257 x 768]
    wpe.weight          [1024 x 768]
  /layers
    /h.0                (block 0)
      /attn
        c_attn.weight   [768 x 2304]
        c_attn.bias     [2304]
        c_proj.weight   [768 x 768]
        c_proj.bias     [768]
      /mlp
        c_fc.weight     [768 x 3072]
        c_fc.bias       [3072]
        c_proj.weight   [3072 x 768]
        c_proj.bias     [768]
      /norms
        ln_1.weight     [768]
        ln_1.bias       [768]
        ln_2.weight     [768]
        ln_2.bias       [768]
    /h.1                (block 1)
      ...same structure...
    ...through h.11...
  /final
    ln_f.weight         [768]
    ln_f.bias           [768]
```

In [4]:
# List all tensors, grouped by component
total_params = 0
total_bytes = 0

print(f"{'Tensor Name':<35s} {'Shape':<20s} {'Params':>12s} {'MB':>8s}")
print("-" * 78)

for name in sorted(weights.keys()):
    tensor = weights[name]
    params = tensor.size
    mb = tensor.nbytes / (1024 * 1024)
    total_params += params
    total_bytes += tensor.nbytes
    print(f"{name:<35s} {str(tensor.shape):<20s} {params:>12,} {mb:>7.2f}")

print("-" * 78)
print(f"{'TOTAL':<35s} {'':<20s} {total_params:>12,} {total_bytes/(1024**2):>7.2f}")
print(f"\nThat's {total_params/1e6:.1f}M parameters = {total_bytes/(1024**2):.0f} MB in float32")

Tensor Name                         Shape                      Params       MB
------------------------------------------------------------------------------
h.0.attn.bias                       (1, 1, 1024, 1024)      1,048,576    4.00
h.0.attn.c_attn.bias                (2304,)                     2,304    0.01
h.0.attn.c_attn.weight              (768, 2304)             1,769,472    6.75
h.0.attn.c_proj.bias                (768,)                        768    0.00
h.0.attn.c_proj.weight              (768, 768)                589,824    2.25
h.0.ln_1.bias                       (768,)                        768    0.00
h.0.ln_1.weight                     (768,)                        768    0.00
h.0.ln_2.bias                       (768,)                        768    0.00
h.0.ln_2.weight                     (768,)                        768    0.00
h.0.mlp.c_fc.bias                   (3072,)                     3,072    0.01
h.0.mlp.c_fc.weight                 (768, 3072)             2,

## Step 5: Where Do the Parameters Live?

Not all parts of the model are equal. Let's see which components use the most parameters.

This is like running `du -sh` on the model's filesystem — where is the disk space going?

In [5]:
# Group parameters by component type
groups = {
    "Token Embeddings (wte)": 0,
    "Position Embeddings (wpe)": 0,
    "Attention (c_attn + c_proj)": 0,
    "FFN / MLP (c_fc + mlp.c_proj)": 0,
    "Layer Norms (ln_1, ln_2, ln_f)": 0,
}

for name, tensor in weights.items():
    p = tensor.size
    if name == "wte.weight":
        groups["Token Embeddings (wte)"] += p
    elif name == "wpe.weight":
        groups["Position Embeddings (wpe)"] += p
    elif "attn" in name:
        groups["Attention (c_attn + c_proj)"] += p
    elif "mlp" in name:
        groups["FFN / MLP (c_fc + mlp.c_proj)"] += p
    elif "ln" in name:
        groups["Layer Norms (ln_1, ln_2, ln_f)"] += p

print("Parameter Distribution (like `du -sh /model/*/`)")
print("=" * 60)

for group, count in groups.items():
    pct = count / total_params * 100
    bar = "#" * int(pct / 2)
    print(f"  {group:<38s} {count:>10,}  {pct:>5.1f}%  {bar}")

print()
print("Key observations:")
print("  - FFN (MLP) is the largest component — ~2/3 of each block's params")
print("  - Attention is the second largest")
print("  - Layer norms are tiny — rounding error")
print("  - Embeddings are shared: wte is also used as the output projection (LM head)")

Parameter Distribution (like `du -sh /model/*/`)
  Token Embeddings (wte)                 38,597,376   28.2%  ##############
  Position Embeddings (wpe)                 786,432    0.6%  
  Attention (c_attn + c_proj)            40,931,328   29.9%  ##############
  FFN / MLP (c_fc + mlp.c_proj)          56,669,184   41.4%  ####################
  Layer Norms (ln_1, ln_2, ln_f)             38,400    0.0%  

Key observations:
  - FFN (MLP) is the largest component — ~2/3 of each block's params
  - Attention is the second largest
  - Layer norms are tiny — rounding error
  - Embeddings are shared: wte is also used as the output projection (LM head)


## Step 6: Look at Actual Weight Values

These tensors are just grids of numbers. Let's look at what the numbers actually are — their range, distribution, and statistics.

This matters for inference because **quantization** (covered later) compresses these numbers to use fewer bits. Understanding the value range tells you how much precision you can afford to lose.

In [6]:
# Inspect a few key tensors
for name in ["wte.weight", "h.0.attn.c_attn.weight", "h.0.mlp.c_fc.weight", "h.11.mlp.c_proj.weight"]:
    t = weights[name]
    print(f"{name}")
    print(f"  Shape:  {t.shape}")
    print(f"  Dtype:  {t.dtype}")
    print(f"  Min:    {t.min():.6f}")
    print(f"  Max:    {t.max():.6f}")
    print(f"  Mean:   {t.mean():.6f}")
    print(f"  Std:    {t.std():.6f}")
    print(f"  First 5 values: {t.flatten()[:5]}")
    print()

wte.weight
  Shape:  (50257, 768)
  Dtype:  float32
  Min:    -1.269817
  Max:    1.785156
  Mean:   0.000380
  Std:    0.143696
  First 5 values: [-0.11010301 -0.03926672  0.03310751  0.13382645 -0.04847569]

h.0.attn.c_attn.weight
  Shape:  (768, 2304)
  Dtype:  float32
  Min:    -2.843634
  Max:    2.795630
  Mean:   0.000053
  Std:    0.199620
  First 5 values: [-0.4738484  -0.26136586 -0.09780374 -0.34988934  0.22432406]

h.0.mlp.c_fc.weight
  Shape:  (768, 3072)
  Dtype:  float32
  Min:    -2.313081
  Max:    4.587719
  Mean:   -0.000749
  Std:    0.141169
  First 5 values: [ 0.09420195  0.09824043 -0.03210221  0.06714731 -0.11544462]

h.11.mlp.c_proj.weight
  Shape:  (3072, 768)
  Dtype:  float32
  Min:    -9.211532
  Max:    9.146134
  Mean:   -0.000435
  Std:    0.198219
  First 5 values: [ 0.06805935  0.09200221 -0.03102017 -0.20520318  0.06265049]



Notice: the values are small (mostly between -0.5 and 0.5), centered near zero, roughly normally distributed. This is typical of trained neural network weights.

The key dimension to remember: **768**. That's the "width" of the model — every hidden state, every intermediate result, is a vector of 768 numbers. Bigger models have wider dimensions (Llama 3.1 8B uses 4096).

## Step 7: The Architecture at a Glance

Now that we've seen every tensor, here's how they fit together:

```
Input: "The cat sat"
         │
         ▼
┌─────────────────┐
│  Tokenize       │  "The" -> 464, "cat" -> 9246, "sat" -> 3332
└────────┬────────┘
         ▼
┌─────────────────┐
│  wte.weight     │  Token IDs -> 768-dim vectors  [50257 x 768]
│  + wpe.weight   │  Add position info             [1024 x 768]
└────────┬────────┘
         ▼
┌─────────────────┐
│  h.0             │  Transformer block 0
│   ln_1 -> attn   │    Norm -> Attention -> Add
│   ln_2 -> mlp    │    Norm -> FFN -> Add
├─────────────────┤
│  h.1             │  Transformer block 1
│   ...            │    (same structure)
├─────────────────┤
│  ...             │  Blocks 2-10
├─────────────────┤
│  h.11            │  Transformer block 11
│   ln_1 -> attn   │
│   ln_2 -> mlp    │
└────────┬────────┘
         ▼
┌─────────────────┐
│  ln_f            │  Final layer norm
└────────┬────────┘
         ▼
┌─────────────────┐
│  wte.weight^T    │  Project back to vocab  [768 x 50257]
│  (weight tying)  │  = logits (one score per vocab token)
└────────┬────────┘
         ▼
   softmax -> sample -> next token
```

Every box is a matrix multiplication (or element-wise op). That's it.  
The entire model is: **embed -> 12 blocks -> norm -> project -> sample -> repeat**.

## Verify: Can We Reconstruct the Parameter Count?

If we truly understand the architecture, we should be able to **calculate** the total parameters from the config alone — without looking at the file.

In [7]:
d = config["n_embd"]      # 768
L = config["n_layer"]     # 12
V = config["vocab_size"]  # 50257
P = config["n_positions"] # 1024
d_ff = 4 * d              # 3072 (GPT-2 convention: 4x expansion)

# Embeddings
emb_params = V * d + P * d  # wte + wpe

# Per transformer block
attn_params = d * 3*d + 3*d + d*d + d   # c_attn (W+b) + c_proj (W+b)
mlp_params  = d * d_ff + d_ff + d_ff*d + d  # c_fc (W+b) + c_proj (W+b)
norm_params = d + d + d + d              # ln_1 (gamma+beta) + ln_2 (gamma+beta)
block_params = attn_params + mlp_params + norm_params

# Final layer norm
final_norm = d + d  # ln_f gamma + beta

# LM head: GPT-2 ties wte.weight, so no extra params
calculated_total = emb_params + L * block_params + final_norm

print(f"Calculated from config:")
print(f"  Embeddings:        {emb_params:>12,}  (wte: {V}x{d} + wpe: {P}x{d})")
print(f"  Per block:         {block_params:>12,}  x {L} blocks = {L * block_params:,}")
print(f"    Attention:       {attn_params:>12,}")
print(f"    FFN/MLP:         {mlp_params:>12,}")
print(f"    Layer norms:     {norm_params:>12,}")
print(f"  Final norm:        {final_norm:>12,}")
print(f"  ─────────────────────────────")
print(f"  Calculated total:  {calculated_total:>12,}")
print(f"  Actual from file:  {total_params:>12,}")
print(f"  Difference:        {total_params - calculated_total:>12,}")
print()

# Explain the difference
causal_mask_params = L * P * P  # 12 layers x 1024 x 1024
print(f"Why the difference?")
print(f"  The file includes {L} causal attention masks (h.N.attn.bias)")
print(f"  Each is [{P} x {P}] = {P*P:,} values  x {L} layers = {causal_mask_params:,}")
print()
print(f"  These are NOT learned parameters — they're precomputed constant masks")
print(f"  (lower-triangular matrix of 1s) that prevent attending to future tokens.")
print(f"  They're baked into the file for convenience, but they're fixed constants.")
print()
print(f"  Calculated + masks = {calculated_total + causal_mask_params:,}")
print(f"  Actual from file   = {total_params:,}")
print(f"  Match: {calculated_total + causal_mask_params == total_params}")
print()
print(f"  The '124M parameter' number people cite for GPT-2 = the learned params only.")

Calculated from config:
  Embeddings:          39,383,808  (wte: 50257x768 + wpe: 1024x768)
  Per block:            7,087,872  x 12 blocks = 85,054,464
    Attention:          2,362,368
    FFN/MLP:            4,722,432
    Layer norms:            3,072
  Final norm:               1,536
  ─────────────────────────────
  Calculated total:   124,439,808
  Actual from file:   137,022,720
  Difference:          12,582,912

Why the difference?
  The file includes 12 causal attention masks (h.N.attn.bias)
  Each is [1024 x 1024] = 1,048,576 values  x 12 layers = 12,582,912

  These are NOT learned parameters — they're precomputed constant masks
  (lower-triangular matrix of 1s) that prevent attending to future tokens.
  They're baked into the file for convenience, but they're fixed constants.

  Calculated + masks = 137,022,720
  Actual from file   = 137,022,720
  Match: True

  The '124M parameter' number people cite for GPT-2 = the learned params only.


## Key Takeaways

- A model file is a **container of weight matrices** — like a tarball of binary arrays with a JSON header.
- SafeTensors format: 8-byte header length + JSON header + raw tensor data. We parsed it with `struct` and `numpy`.
- GPT-2 has **148 tensors** organized into: embeddings, 12 transformer blocks (each with attention + FFN + norms), and a final norm.
- The **FFN holds ~2/3 of each block's parameters**. Attention is second. Layer norms are negligible.
- The magic number is **768** — every hidden state is a 768-dimensional vector. Bigger models = wider vectors.
- Weight values are small floats (~[-0.5, 0.5]). This is why quantization can compress them.

## Next

**Notebook 02** — We build a tokenizer from scratch to convert text into the token IDs that index into `wte.weight`.

## References

- *Inference Engineering* Ch 2.2.1 (pp. 49–50) — LLM Architecture, config.json
- [SafeTensors format spec](https://huggingface.co/docs/safetensors/)